In [1]:
Interesting_regions =   [4, 5, 32, 33, 44, 45, 48, 49]
Interesting_right =   [5, 33, 45, 49]
Interesting_left =   [4, 32, 44, 48]

Good_subjects =         [0, 2, 5, 9, 12, 14]
Ok_subjects =           [1, 3, 4, 6, 11, 13, 19]
Other_subjects = [i for i in range(20) if i not in Ok_subjects and i not in Good_subjects]

Subjects_dict = {"Good_subjects": Good_subjects, "Ok_subjects":Ok_subjects, "Other_subjects":Other_subjects}

data_folder="/Users/giovanni.messuti/Desktop/BCI_Project/Data/"

freq_personalized = {"subject_0":[9,14.5],
                     "subject_1":[8,12],
                     "subject_2":[8,13.5],
                     "subject_3":[4,9.5],
                     "subject_4":[18,23.5],
                     "subject_5":[4,9.5],
                     "subject_6":[4,9.5],
                     "subject_7":[4,9.5],
                     "subject_8":[4,12],
                     "subject_9":[9.5,15.5],
                     "subject_10":[9,14.5],
                     "subject_11":[4,10.5],
                     "subject_12":[8,13.5],
                     "subject_13":[4,10.],
                     "subject_14":[12,18],
                     "subject_15":[4,8],
                     "subject_16":[4,13],
                     "subject_17":[4,12],
                     "subject_18":[4,9.5],
                     "subject_19":[4,12.5],
}

from BCI_Library import read_subject, plot_brain, train_models, extract_features_more_bands, select_regions_Cohen_effect_size
from BCI_Library import saveresults_pickle, select_rows, compute_welch_select_regions, extract_features_from_spectra_more_bands


# jfjff

In [ ]:
#Integrate EEG EMG
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

seed = 224

# keys of the performance dictionary
saveresults = True
filepath = "/Users/giovanni.messuti/Desktop/BCI_Project/Results/BCI_Performances_region_selection_on_folds_8_30_Hz.pkl"
file_input = filepath
file_output = filepath
miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Comments= "Cohen's d effect size selection" # "-"
PFR = False

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

tutte_roi = 68
verbose = False

for DataType in ["EEG concat MEG"]:
    for Classifier_name, meth in zip(["SVC", "RandomForest", "LDA"],[SVC, RandomForestClassifier, LDA]):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Performance": [],
            "Features": [],
            "Comments": [],
            "Date": []
        })
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(range(20)):

            data_2_MI_EEG = read_subject(data_folder, "EEG", "MI",subject_index, verbose=False)
            data_2_Rest_EEG =read_subject(data_folder, "EEG", "Baseline",subject_index, verbose=False)
            data_2_MI_MEG = read_subject(data_folder, "MEG", "MI",subject_index, verbose=False)
            data_2_Rest_MEG =read_subject(data_folder, "MEG", "Baseline",subject_index, verbose=False)

            WMI_EEG, WRe_EEG, FreqMI_EEG = compute_welch(data_2_MI_EEG, data_2_Rest_EEG, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            # WMI.shape (trials, ROIs, freqs)
            labels_EEG = [1 for __ in range(len(WMI_EEG))] + [0 for __ in range(len(WRe_EEG))] 
            Feat_extraction_EEG = extract_features_from_spectra_more_bands(WMI_EEG, WRe_EEG, [ _ for _ in range(tutte_roi)], FreqMI_EEG,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)

            Feat_MI_EEG, Feat_Rest_EEG, labls_MI_EEG, labls_Rest_EEG, all_regions_EEG = Feat_extraction_EEG
            # Feat_MI.shape (Trials, selected_ROIs, features for Roi)
            Feat_All_EEG = np.concatenate((Feat_MI_EEG, Feat_Rest_EEG), axis=0)
            labels_All_EEG= labels_EEG

            #################################################################################

            WMI_MEG, WRe_MEG, FreqMI_MEG = compute_welch(data_2_MI_MEG, data_2_Rest_MEG, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            labels_MEG = [1 for __ in range(len(WMI_MEG))] + [0 for __ in range(len(WRe_MEG))] 
            Feat_extraction_MEG = extract_features_from_spectra_more_bands(WMI_MEG, WRe_MEG, [ _ for _ in range(tutte_roi)], FreqMI_MEG,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)
            Feat_MI_MEG, Feat_Rest_MEG, labls_MI_MEG, labls_Rest_MEG, all_regions_MEG = Feat_extraction_MEG
            Feat_All_MEG = np.concatenate((Feat_MI_MEG, Feat_Rest_MEG), axis=0)
            labels_All_MEG= labels_MEG


            for num_best_ROIs in [3,5,10,20,40]:

                fold_accuracies = []

                for fold_idx, (train_idx, val_idx) in enumerate(kf.split([0 for _ in range(len(labels_MEG))], labels_MEG)):

                    # selecting region based on training data only
                    training_index_MI = train_idx[np.where(train_idx<len(WMI_EEG))]
                    training_index_Re = train_idx[np.where(train_idx>=len(WMI_EEG))]-len(WMI_EEG)

                    regions_selected_EEG, effect_size = select_regions_Cohen_effect_size(wpsdMI=WMI_EEG[training_index_MI], 
                                                                    wpsdRest=WRe_EEG[training_index_MI], 
                                                                    important_regions=[], nbest_regions=num_best_ROIs)
                    
                    regions_selected_MEG, effect_size = select_regions_Cohen_effect_size(wpsdMI=WMI_MEG[training_index_MI], 
                                                                    wpsdRest=WRe_MEG[training_index_MI], 
                                                                    important_regions=[], nbest_regions=num_best_ROIs)

                    
                    Feat_All_regions_selected_EEG = Feat_All_EEG[:,regions_selected_EEG,:].reshape(Feat_All_EEG.shape[0],-1)
                    Feat_All_regions_selected_MEG = Feat_All_MEG[:,regions_selected_MEG,:].reshape(Feat_All_MEG.shape[0],-1)
                    # Feat_All_regions_selected_EEG.shape is (trials,features)
                    Feat_All_regions_selected_concat = np.concatenate((Feat_All_regions_selected_EEG,Feat_All_regions_selected_MEG),axis=-1)
                    #####################################################################################################################
                    acc, cm, model = train_single_model(Feat_All_regions_selected_concat, np.array(labels_All_MEG),
                                                        train_idx, val_idx, method=meth, seed=224)

                    fold_accuracies.append(acc)
                    if verbose:
                        print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                        print(cm)

                    #####################################################################################################################

                # Summary
                if verbose:
                    print("\n==================================")
                    print("Mean accuracy:", np.mean(fold_accuracies))
                    print("Std accuracy:", np.std(fold_accuracies))



                new_row = {
                    "ROIs": [[regions_selected_EEG,regions_selected_MEG]],
                    "NROIs":[len(regions_selected_EEG)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Performance": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        


        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)